# Unsupervised Learning & Feature Reduction on UCI HAR

Dataset: *Human Activity Recognition Using Smartphones* (UCI) — 561 engineered features computed from
2.56 s windows of smartphone accelerometer/gyroscope signals, 30 subjects, 6 activities.

**Protocol.** The provided training set is split by subject into **train** and **dev**; all exploration,
reduction and model selection use only these. The provided **test** set is loaded and scaled in Phase 1
(with the scaler fitted on train only) and cached, but it is not analysed, plotted or scored until the
single final evaluation in Phase 5. Activity labels are never given to the unsupervised methods — they are
only used afterwards to evaluate them.

All figures are saved to `figures/` (PNG, dpi=200), all numeric results to `results/` (CSV), and the
processed arrays to `cache/`.

## Phase 1 - Setup

The cells below import the libraries, fix `random_state=42`, and define the shared helpers (locating and
loading the dataset, saving figures/tables, the subject-wise split, the cache). They then load the provided
train and test sets, split the provided training set into train/dev **by subject**, fit a `StandardScaler`
on train only, transform train/dev/test with it, and cache the processed arrays to `cache/har_processed.npz`
so later phases load them quickly.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path.cwd()
FIG_DIR = PROJECT_DIR / "figures"
RES_DIR = PROJECT_DIR / "results"
CACHE_DIR = PROJECT_DIR / "cache"
CACHE_FILE = CACHE_DIR / "har_processed.npz"
for d in (FIG_DIR, RES_DIR, CACHE_DIR):
    d.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [2]:
# ---------- Shared helpers: locating and loading the dataset ----------

def find_data_dir(root=PROJECT_DIR):
    """Find the extracted 'UCI HAR Dataset' directory (has train/, test/, features.txt, activity_labels.txt).

    Searches the project folder; ignores macOS '__MACOSX' metadata copies. Nothing is downloaded or extracted.
    """
    for features_file in sorted(root.rglob("features.txt")):
        d = features_file.parent
        if "__MACOSX" in d.parts:
            continue
        if (d / "activity_labels.txt").exists() and (d / "train").is_dir() and (d / "test").is_dir():
            return d
    raise FileNotFoundError(f"No 'UCI HAR Dataset' directory found under {root}")


DATA_DIR = find_data_dir()
print("Dataset directory:", DATA_DIR.relative_to(PROJECT_DIR))


def load_activity_labels():
    """Map integer activity code -> activity name."""
    df = pd.read_csv(DATA_DIR / "activity_labels.txt", sep=r"\s+", header=None, names=["code", "activity"])
    return dict(zip(df["code"], df["activity"]))


def load_feature_names():
    """Return the 561 feature names, made unique (features.txt repeats some band-energy names)."""
    names = pd.read_csv(DATA_DIR / "features.txt", sep=r"\s+", header=None, names=["idx", "name"])["name"]
    counts, unique = {}, []
    for n in names:
        counts[n] = counts.get(n, 0) + 1
        unique.append(n if counts[n] == 1 else f"{n}_{counts[n]}")
    return unique


ACTIVITY_LABELS = load_activity_labels()
ACTIVITY_ORDER = [ACTIVITY_LABELS[k] for k in sorted(ACTIVITY_LABELS)]
FEATURE_NAMES = load_feature_names()


def load_split(split):
    """Load one provided split ('train' or 'test').

    Returns X (DataFrame, 561 named features), y (Series of activity NAMES), subject (int Series).
    """
    assert split in ("train", "test")
    d = DATA_DIR / split
    X = pd.read_csv(d / f"X_{split}.txt", sep=r"\s+", header=None, names=FEATURE_NAMES)
    codes = pd.read_csv(d / f"y_{split}.txt", header=None)[0]
    y = pd.Series(pd.Categorical(codes.map(ACTIVITY_LABELS), categories=ACTIVITY_ORDER), name="activity")
    subject = pd.read_csv(d / f"subject_{split}.txt", header=None)[0].rename("subject")
    return X, y, subject


# ---------- Shared helpers: saving outputs ----------

def save_fig(fig, name):
    """Save a figure to figures/<name>.png at dpi=200 (it is also shown inline)."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"saved {path.relative_to(PROJECT_DIR)}")


def save_table(df, name, index=True):
    """Save a table to results/<name>.csv."""
    path = RES_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"saved {path.relative_to(PROJECT_DIR)}")


# ---------- Shared helpers: splitting ----------

def subject_split(X, y, subject, dev_size=0.2):
    """Split by SUBJECT with GroupShuffleSplit (~dev_size of subjects to dev) so no person is in both."""
    gss = GroupShuffleSplit(n_splits=1, test_size=dev_size, random_state=RANDOM_STATE)
    tr_idx, dev_idx = next(gss.split(X, y, groups=subject))
    take = lambda obj, idx: obj.iloc[idx].reset_index(drop=True)
    return (take(X, tr_idx), take(y, tr_idx), take(subject, tr_idx),
            take(X, dev_idx), take(y, dev_idx), take(subject, dev_idx))


# ---------- Shared helpers: processed-data cache ----------

def load_cache(path=CACHE_FILE):
    """Load the processed arrays written in Phase 1.

    Returns a dict with X_{train,dev,test} (standardized float arrays), y_{train,dev,test} (activity names),
    subject_{train,dev,test}, feature_names, activity_order, scaler_mean, scaler_scale.
    """
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

Dataset directory: human+activity+recognition+using+smartphones\UCI HAR Dataset\UCI HAR Dataset


In [3]:
# Load the provided train and test sets
t0 = time.perf_counter()
X_train_full, y_train_full, subject_train_full = load_split("train")
X_test, y_test, subject_test = load_split("test")
print(f"Loaded in {time.perf_counter() - t0:.1f}s")
print(f"Provided train: X={X_train_full.shape}, {subject_train_full.nunique()} subjects, "
      f"{X_train_full.isna().sum().sum()} missing values")
print(f"Provided test:  X={X_test.shape}, {subject_test.nunique()} subjects, "
      f"{X_test.isna().sum().sum()} missing values")
print(f"Features: {len(FEATURE_NAMES)} ({len(set(FEATURE_NAMES))} unique after de-duplication); "
      f"activities: {ACTIVITY_ORDER}")
X_train_full.iloc[:5, :8]

Loaded in 1.4s
Provided train: X=(7352, 561), 21 subjects, 0 missing values
Provided test:  X=(2947, 561), 9 subjects, 0 missing values
Features: 561 (561 unique after de-duplication); activities: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672


In [4]:
# Split the provided training set into train / dev BY SUBJECT
X_train, y_train, subject_train, X_dev, y_dev, subject_dev = subject_split(
    X_train_full, y_train_full, subject_train_full, dev_size=0.2)

assert set(subject_train).isdisjoint(subject_dev), "subject leakage between train and dev"
assert set(subject_train_full).isdisjoint(subject_test), "provided train/test share subjects"

for name, s in [("train", subject_train), ("dev", subject_dev), ("test", subject_test)]:
    print(f"{name:5s}: {len(s):5d} rows, {s.nunique():2d} subjects -> {sorted(s.unique().tolist())}")

split_summary = pd.DataFrame({
    "train": y_train.value_counts().reindex(ACTIVITY_ORDER),
    "dev": y_dev.value_counts().reindex(ACTIVITY_ORDER),
    "test": y_test.value_counts().reindex(ACTIVITY_ORDER),
})
split_summary.index.name = "activity"
split_summary.loc["TOTAL"] = split_summary.sum()
for c in ["train", "dev", "test"]:
    split_summary[f"{c}_%"] = (100 * split_summary[c] / split_summary.loc["TOTAL", c]).round(1)
split_summary.loc["n_subjects", ["train", "dev", "test"]] = [
    subject_train.nunique(), subject_dev.nunique(), subject_test.nunique()]
split_summary[["train", "dev", "test"]] = split_summary[["train", "dev", "test"]].astype(int)

save_table(split_summary, "phase1_split_summary")
display(split_summary)

train:  5551 rows, 16 subjects -> [5, 6, 7, 8, 11, 14, 16, 17, 19, 21, 22, 23, 26, 28, 29, 30]
dev  :  1801 rows,  5 subjects -> [1, 3, 15, 25, 27]
test :  2947 rows,  9 subjects -> [2, 4, 9, 10, 12, 13, 18, 20, 24]
saved results\phase1_split_summary.csv


,train,dev,test,train_%,dev_%,test_%
activity,,,,,,
WALKING,888,338,496,16.0,18.8,16.8
WALKING_UPSTAIRS,797,276,471,14.4,15.3,16.0
WALKING_DOWNSTAIRS,744,242,420,13.4,13.4,14.3
SITTING,993,293,491,17.9,16.3,16.7
STANDING,1053,321,532,19.0,17.8,18.1
LAYING,1076,331,537,19.4,18.4,18.2
TOTAL,5551,1801,2947,100.0,100.0,100.0
n_subjects,16,5,9,NaN,NaN,NaN


In [5]:
# Fit StandardScaler on TRAIN ONLY, then transform train, dev and test with it
scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)
Xs_dev = scaler.transform(X_dev)
Xs_test = scaler.transform(X_test)

print(f"train after scaling: mean={Xs_train.mean():+.2e}, std={Xs_train.std():.3f}")
print(f"dev   after scaling: mean={Xs_dev.mean():+.3f}, std={Xs_dev.std():.3f}  (not exactly 0/1: scaler fit on train)")
print(f"Zero-variance features in train: {(scaler.scale_ == 0).sum()}")

train after scaling: mean=+4.06e-19, std=1.000
dev   after scaling: mean=-0.048, std=0.901  (not exactly 0/1: scaler fit on train)
Zero-variance features in train: 0


In [6]:
# Cache the processed arrays so later phases load them quickly
t0 = time.perf_counter()
np.savez_compressed(
    CACHE_FILE,
    X_train=Xs_train.astype(np.float32), X_dev=Xs_dev.astype(np.float32), X_test=Xs_test.astype(np.float32),
    y_train=y_train.to_numpy(dtype=str), y_dev=y_dev.to_numpy(dtype=str), y_test=y_test.to_numpy(dtype=str),
    subject_train=subject_train.to_numpy(), subject_dev=subject_dev.to_numpy(), subject_test=subject_test.to_numpy(),
    feature_names=np.array(FEATURE_NAMES), activity_order=np.array(ACTIVITY_ORDER),
    scaler_mean=scaler.mean_, scaler_scale=scaler.scale_,
)
print(f"Wrote {CACHE_FILE.relative_to(PROJECT_DIR)} ({CACHE_FILE.stat().st_size / 1e6:.1f} MB) "
      f"in {time.perf_counter() - t0:.1f}s")

# Round-trip check
t0 = time.perf_counter()
data = load_cache()
print(f"Reloaded in {time.perf_counter() - t0:.2f}s:",
      {k: v.shape for k, v in data.items() if k.startswith(("X_", "y_", "subject_"))})
assert np.allclose(data["X_train"], Xs_train, atol=1e-5) and (data["y_test"] == y_test.to_numpy(dtype=str)).all()

Wrote cache\har_processed.npz (19.7 MB) in 0.7s
Reloaded in 0.10s: {'X_train': (5551, 561), 'X_dev': (1801, 561), 'X_test': (2947, 561), 'y_train': (5551,), 'y_dev': (1801,), 'y_test': (2947,), 'subject_train': (5551,), 'subject_dev': (1801,), 'subject_test': (2947,)}
